# Andrew Sparkes - DSAI201 Section 01 - Project 4 - Barry Bonds Analysis

In [ ]:
# setup and imports
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.mlab as mlab
import numpy as np
from datetime import datetime
from scipy import stats

print("Setup complete.")


Setup complete.


In [2]:
# User config settings
alpha = 0.05  # significance level


# Hypothesis Testing with Barry Bonds
Barry Bonds holds the MLB record for most career home runs (and is generally considered to be one of the greatest hitters of all-time).
His career, however, has been tainted by steroid allegations and he remains left out of the Hall of Fame.
According to reddit user u/reptheevt, the general opinion is that Bonds began using steroids in the time leading up to the 1999 MLB season.
For this project, we will treat **1987-1998** as Bonds' "pre-steroid" era and **1999-2007** as his "post-steroid" era.

Your goal is to investigate whether there is statistical evidence for a difference in his performance between the two eras of his career.
We will compare the two eras using two metric:
* Home runs per game = HR/G
* OPS (On-Base Percentage plus Slugging Percentage), a simple indicator of an impactful hitter. OPS

**This project is intentionally loosely structured.** You are responsible for an analysis that is coherent, justified, and statistically sound.
Your notebook should demonstrate the following:

**Data Collection**
* You will need to find season-level statistics for Bonds' (the website [baseball reference ](https://www.baseball-reference.com/) might be useful)
* You will have to manage and organize the data in Python
* Include any datasets you download as part of your submission.

In [ ]:
# Set the path to the file you'd like to load
file_path = "data.csv"  # downloaded from baseball stats website, citation in csv

# get it into a df
df = pd.read_csv(file_path, header=0, comment="-")

print(df.head(5))  # sanity check, make sure we're getting valid data back out

# do any data wrangling/cleaning/pruning as needed (AND AS DESCRIBED IN METHODS SECTION)
print(
    f"Thankfully, this data was well prepared and curated, and required no additional cleaning or wrangling for our purposes."
)


   Season  Age Team  Lg  WAR    G   PA   AB    R    H  ...   rOBA  Rbat+   TB  \
0    1986   21  PIT  NL  3.5  113  484  413   72   92  ...  0.358    106  172   
1    1987   22  PIT  NL  5.8  150  611  551   99  144  ...  0.369    116  271   
2    1988   23  PIT  NL  6.3  144  614  538   97  152  ...  0.387    152  264   
3    1989   24  PIT  NL  8.0  159  679  580   96  144  ...  0.356    124  247   
4    1990   25  PIT  NL  9.7  151  621  519  104  156  ...  0.435    171  293   

   GIDP  HBP  SH  SF  IBB     Pos  Awards  
0     4    2   2   2    2    *8/H     ROY  
1     4    3   0   3    3  *78H/9     NaN  
2     3    2   0   2   14   *7H/8     NaN  
3     9    1   1   4   22    *7/H     NaN  
4     8    3   0   6   15   *7/H8   ASMVP  

[5 rows x 33 columns]
Thankfully, this data was well prepared and curated, and required no additional cleaning or wrangling for our purposes.


**Hypothesis Testing**
* Appropriately collect the data and metrics for the two periods.
  Downloaded Barry Bonds' season-level statistics from Baseball Reference and organized them into a pandas DataFrame:


In [ ]:
# get a df with the pre-steroid years only, and another with the post-steroid years only
presteroid_data = df[df["Season"].astype(int).between(1987, 1998)]
poststeroid_data = df[df["Season"].astype(int).between(1999, 2007)]


# aliasing these for ease of use later
pre_HRG = presteroid_data["HR"] / presteroid_data["G"]
pre_OPS = presteroid_data["OPS"]
post_HRG = poststeroid_data["HR"] / poststeroid_data["G"]
post_OPS = poststeroid_data["OPS"]


# compute means, will likely use later
pre_HRG_mean = np.mean(pre_HRG)
post_HRG_mean = np.mean(post_HRG)

pre_OPS_mean = np.mean(pre_OPS)
post_OPS_mean = np.mean(post_OPS)


# compute variances of all sets for later
pre_HRG_var = np.var(pre_HRG)
post_HRG_var = np.var(post_HRG)

pre_OPS_var = np.var(pre_OPS)
post_OPS_var = np.var(post_OPS)


In [ ]:
# sanity check, make sure the first few rows look right
print(presteroid_data.head())
print(poststeroid_data.head())


   Season  Age Team  Lg  WAR    G   PA   AB    R    H  ...   rOBA  Rbat+   TB  \
1    1987   22  PIT  NL  5.8  150  611  551   99  144  ...  0.369    116  271   
2    1988   23  PIT  NL  6.3  144  614  538   97  152  ...  0.387    152  264   
3    1989   24  PIT  NL  8.0  159  679  580   96  144  ...  0.356    124  247   
4    1990   25  PIT  NL  9.7  151  621  519  104  156  ...  0.435    171  293   
5    1991   26  PIT  NL  8.0  153  634  510   95  149  ...  0.413    156  262   

   GIDP  HBP  SH  SF  IBB     Pos  Awards  
1     4    3   0   3    3  *78H/9     NaN  
2     3    2   0   2   14   *7H/8     NaN  
3     9    1   1   4   22    *7/H     NaN  
4     8    3   0   6   15   *7/H8   ASMVP  
5     8    4   0  13   25   *7/H8     MVP  

[5 rows x 33 columns]
    Season  Age Team  Lg   WAR    G   PA   AB    R    H  ...   rOBA  Rbat+  \
13    1999   34  SFG  NL   3.8  102  434  355   91   93  ...  0.435    158   
14    2000   35  SFG  NL   7.7  143  607  480  129  147  ...  0.465   

* Clearly stated null and alternative hypotheses.
- $H_0$ = null hypothesis: There is no difference in Barry Bonds' performance metrics (HR/G and OPS) between the pre-steroid era (1987-1998) and post-steroid era (1999-2007).
- $H_a$ = alternative hypothesis: There is a significant difference in Barry Bonds' performance metrics (HR/G and OPS) between the pre-steroid era (1987-1998) and post-steroid era (1999-2007).

Note, technically, I should probably break this into two completely separate test hypothese for HR/G and OPS as separate metrics, but for simplicity, I am using the same statements, just swapping in either HR/G or OPS in as needed.


In [6]:
print(f"Pre-steroid-------------")
print(f"HR/G Mean: {pre_HRG_mean:.6f} | OPS Mean: {pre_OPS_mean:.6f}")
print(f"Post-steroid------------")
print(f"HR/G Mean: {post_HRG_mean:.6f} | OPS Mean: {post_OPS_mean:.6f}")


Pre-steroid-------------
HR/G Mean: 0.223420 | OPS Mean: 0.983583
Post-steroid------------
HR/G Mean: 0.322937 | OPS Mean: 1.189778


* Check and address assumptions of each test (e.g., the approximate independence and normality assumptions associated with t-tests)


In [ ]:
print(
    f"Well, for ease of testing, we'll be assuming that these game stats are independent of one another. This is a reasonable assumption because players performance largely will not reflect upon the next performance in a statistically significant way."
)


Well, for ease of testing, we'll be assuming that these game stats are independent of one another. This is a reasonable assumption because players performance largely will not reflect upon the next performance in a statistically significant way.


* Make sure that both data sets are normally distributed, as well as having equal variances.

In [22]:
print(f"Pre-steroid-------------")
print(f"HR/G Variance: {pre_HRG_var:.6f} | OPS Variance: {pre_OPS_var:.6f}")
print(f"Post-steroid------------")
print(f"HR/G Variance: {post_HRG_var:.6f} | OPS Variance: {post_OPS_var:.6f}")
print("-" * 80)

print(
    f"We find that the variances for HR/G and OPS from the pre-steroid era compared to the variances for the post-steroid period are not equal, but are within a 4:1 ratio, so the student's 2-sample t-test is appropriate to use here, instead of utilizing Welch's t-test."
)


# compare both sets to normal distribution
# quick helper function to barf out normal test results
def compareToNormal(data):
    res = stats.normaltest(data)
    print(
        f"Normality test result: Statistic={res.statistic:.6f}, p-value={res.pvalue:.6f}"
    )
    if res.pvalue < alpha:
        print("The data significantly deviates from a normal distribution.")
    else:
        print("The data does not significantly deviate from a normal distribution.")
    print("-" * 80)


print("-" * 80)
print(f"pre-steroid HR/G normality test:")
compareToNormal(pre_HRG)

print("-" * 80)
print(f"post-steroid HR/G normality test:")
compareToNormal(post_HRG)

print("-" * 80)
print(f"pre-steroid OPS normality test:")
compareToNormal(pre_OPS)

print("-" * 80)
print(f"post-steroid OPS normality test:")
compareToNormal(post_OPS)


Pre-steroid-------------
HR/G Variance: 0.003320 | OPS Variance: 0.012029
Post-steroid------------
HR/G Variance: 0.005725 | OPS Variance: 0.027029
--------------------------------------------------------------------------------
We find that the variances for HR/G and OPS from the pre-steroid era compared to the variances for the post-steroid period are not equal, but are within a 4:1 ratio, so the student's 2-sample t-test is appropriate to use here, instead of utilizing Welch's t-test.
--------------------------------------------------------------------------------
pre-steroid HR/G normality test:
Normality test result: Statistic=0.025327, p-value=0.987416
The data does not significantly deviate from a normal distribution.
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
post-steroid HR/G normality test:
Normality test result: Statistic=1.107393, p-value=0.574821
The data 

* Conduct separate **two-sample t-test** for the two metrics used in our study.

In [16]:
HRG_t_statistic, HRG_p_value = stats.ttest_ind(pre_HRG, post_HRG, equal_var=True)

OPS_t_statistic, OPS_p_value = stats.ttest_ind(pre_OPS, post_OPS, equal_var=True)

print(f"{HRG_t_statistic=:.6f}, {HRG_p_value=:.6f}")
print(f"{OPS_t_statistic=:.6f}, {OPS_p_value=:.6f}")


HRG_t_statistic=-3.254680, HRG_p_value=0.004170
OPS_t_statistic=-3.273863, OPS_p_value=0.003994


**Scientific communication**
* State your $p$-values and the conclusions based on these $p$-values.

In [17]:
# output p-values
print(f"The p-value for the 2-sample t-test for HR/G is {HRG_p_value:.6f}.")
print(f"The p-value for the 2-sample t-test for OPS is {OPS_p_value:.6f}.")


# output conclusions based on p-values
if HRG_p_value < alpha:
    print("\nDecision: Reject the null hypothesis for HR/G winning rate.")
    print("There is a significant difference between the means of the two samples.")
else:
    print("\nDecision: Fail to reject the null hypothesis for HR/G winning rate.")
    print("There is no significant difference between the means of the two samples.")


if OPS_p_value < alpha:
    print("\nDecision: Reject the null hypothesis for OPS.")
    print("There is a significant difference between the means of the two samples.")
else:
    print("\nDecision: Fail to reject the null hypothesis for OPS.")
    print("There is no significant difference between the means of the two samples.")


The p-value for the 2-sample t-test for HR/G is 0.004170.
The p-value for the 2-sample t-test for OPS is 0.003994.

Decision: Reject the null hypothesis for HR/G winning rate.
There is a significant difference between the means of the two samples.

Decision: Reject the null hypothesis for OPS.
There is a significant difference between the means of the two samples.


* Place your conclusion in the context of Bonds' performance and his candidacy for the Hall of Fame (trust me, his statistics are Hall of Fame worthy)

# Conclusion


To recap, we first collected data for Barry Bonds' performance metrics (HR/G and OPS) from Baseball Reference, organizing it into a pandas DataFrame. We then formulated our null and alternative hypotheses regarding the differences in these metrics (home runs/total games, and OPS) between the pre-steroid (1987 - 1998) and post-steroid (1999 - 2007) eras. We checked the assumptions of normality and approximately equal variances for both datasets, confirming that they were met. Note: Our variances were not exactly equal, but close enough to proceed with the t-tests within the 4:1 tolerance, and should not require utilizing Welch's t-test instead. 

We performed separate two-sample t-tests for both HR/G and OPS metrics on our data and then analyzed the results. With our alpha level set at $0.05$, we found that the p-values for both metrics were significantly below this threshold, leading us to reject the null hypothesis in both cases. This indicates that there is a statistically significant difference in Barry Bonds' performance metrics between the pre-steroid and post-steroid eras. Specifically, we observed an increase in both HR/G and OPS during the post-steroid era, suggesting an improvement in his performance, but not necessarily proving anything definitively. My interpretation of the results is that while there is a statistical difference, it does not necessarily imply causation or confirm steroid use as the definitive factor for the performance change. As for Barry Bonds' candidacy for the Hall of Fame, his statistics remain impressive regardless of the era, and while the steroid allegations may cast doubt on the legitimacy of his performance, his metrics alone are worthy of consideration for Hall of Fame induction, steroids or not.